# Step 5 — The 'Moron Premium' as a Level (Spread) Phenomenon

Measures UK–Germany and UK–peer-average spreads at 10y and 30y, dates the May-2026 widening, and decomposes it into shared-non-euro vs UK-specific.

**Script notes**

```
Step 5 - The 'moron premium' as a LEVEL (spread) phenomenon
PCA (steps 2-4) used daily CHANGES and deliberately differenced away levels.
The article's 'moron premium' is a level concept: UK pays EXTRA yield vs peers.
Here we measure spreads (not co-movement) at 10y and 30y.

Spreads:
  UK - DE            (classic safe-haven / risk-premium benchmark; article's core comparison)
  UK - peer_avg      (peer_avg = mean of US, DE, FR, IT; isolates UK-specific vs whole set)
Cross-check:
  US - DE            (is widening an Anglosphere story or uniquely UK?)

Episodes:
  pre-war baseline   : Jan-Feb 2026 mean (before US-Iran war ~26 Feb)
  May-2026 peak      : max spread in May 2026
  decompose change   : how much of UK-DE widening is shared with US (US-DE) vs UK-specific
```

> Paths assume this notebook sits in a folder with a `./data/` subfolder holding the raw downloads, and that earlier steps have written their intermediate CSVs into the same folder. Run `load_yields` first.

In [1]:
import numpy as np
import pandas as pd

In [2]:
lv = pd.read_csv("./levels_aligned.csv", index_col=0, parse_dates=True)

In [3]:
def spreads(tenor):
    uk = lv[f"UK_{tenor}"]; de = lv[f"DE_{tenor}"]; us = lv[f"US_{tenor}"]
    fr = lv[f"FR_{tenor}"]; it = lv[f"IT_{tenor}"]
    peer_avg = lv[[f"US_{tenor}", f"DE_{tenor}", f"FR_{tenor}", f"IT_{tenor}"]].mean(axis=1)
    return pd.DataFrame({
        "UK_minus_DE":   (uk - de)*100,        # bp
        "UK_minus_peer": (uk - peer_avg)*100,
        "US_minus_DE":   (us - de)*100,
    })

In [4]:
for tenor in ["10y", "30y"]:
    sp = spreads(tenor)
    print("="*68)
    print(f"LEVEL SPREADS  ({tenor})   units: basis points")
    print("="*68)

LEVEL SPREADS  (10y)   units: basis points
LEVEL SPREADS  (30y)   units: basis points


In [5]:
    pre  = sp.loc["2026-01-01":"2026-02-25"].mean()
    maypk= sp.loc["2026-05-01":"2026-05-29"].max()
    mayd = sp.loc["2026-05-01":"2026-05-29"].idxmax()
    full_mean = sp.mean()

In [6]:
    out = pd.DataFrame({
        "full_sample_mean": full_mean,
        "pre_war_mean(Jan-Feb26)": pre,
        "May26_peak": maypk,
        "widening(pre->peak)": maypk - pre,
    })
    print("\n", out.round(1).to_string())
    print("\n May-2026 peak dates:")
    for c in sp.columns:
        print(f"   {c:14s} peak {maypk[c]:6.1f}bp on {mayd[c].date()}")


                full_sample_mean  pre_war_mean(Jan-Feb26)  May26_peak  widening(pre->peak)
UK_minus_DE               217.6                    187.5       230.8                 43.3
UK_minus_peer             114.9                    109.4       142.8                 33.5
US_minus_DE               170.1                    133.8       151.0                 17.2

 May-2026 peak dates:
   UK_minus_DE    peak  230.8bp on 2026-05-15
   UK_minus_peer  peak  142.8bp on 2026-05-15
   US_minus_DE    peak  151.0bp on 2026-05-26


In [7]:
    # decomposition of UK-DE widening into shared-with-US vs UK-specific
    d_ukde = (maypk["UK_minus_DE"] - pre["UK_minus_DE"])
    d_usde = (maypk["US_minus_DE"] - pre["US_minus_DE"])
    uk_specific = d_ukde - d_usde
    print(f"\n Decomposition of UK-DE widening ({tenor}, pre-war -> May peak):")
    print(f"   total UK-DE widening      : {d_ukde:6.1f} bp")
    print(f"   shared 'non-euro' (US-DE) : {d_usde:6.1f} bp  ({d_usde/d_ukde*100 if d_ukde else 0:4.0f}% of total)")
    print(f"   UK-SPECIFIC residual      : {uk_specific:6.1f} bp  ({uk_specific/d_ukde*100 if d_ukde else 0:4.0f}% of total)")


 Decomposition of UK-DE widening (30y, pre-war -> May peak):
   total UK-DE widening      :   43.3 bp
   shared 'non-euro' (US-DE) :   17.2 bp  (  40% of total)
   UK-SPECIFIC residual      :   26.1 bp  (  60% of total)


In [8]:
    sp.to_csv(f"./spreads_{tenor}.csv")
    print(f"\n saved spreads_{tenor}.csv\n")


 saved spreads_30y.csv



In [9]:
# IS THE PREMIUM A LONG-END STORY?  UK-DE: 10y vs 30y
print("="*68)
print("IS THE PREMIUM A LONG-END STORY?  UK-DE: 10y vs 30y")
print("="*68)
s10 = spreads("10y")["UK_minus_DE"]
s30 = spreads("30y")["UK_minus_DE"]
comp = pd.DataFrame({"UK-DE 10y": s10, "UK-DE 30y": s30})
print("\n full-sample mean :", comp.mean().round(1).to_dict())
print(" May-2026 peak    :", comp.loc["2026-05-01":"2026-05-29"].max().round(1).to_dict())
print(" latest (29 May)  :", comp.iloc[-1].round(1).to_dict())

IS THE PREMIUM A LONG-END STORY?  UK-DE: 10y vs 30y

 full-sample mean : {'UK-DE 10y': 185.9, 'UK-DE 30y': 217.6}
 May-2026 peak    : {'UK-DE 10y': 210.4, 'UK-DE 30y': 230.8}
 latest (29 May)  : {'UK-DE 10y': 191.0, 'UK-DE 30y': 212.1}
